# Understanding the Embedding Fix in dHT

This notebook explains the "embedding fix" mechanism in dHT and how it enables backward compatibility with pre-trained ViT models.

## The Challenge

Standard ViT models expect:
- Fixed number of patches (e.g., 14×14 = 196 patches for 224×224 images)
- Each patch represented as a flattened vector (e.g., 16×16×3 = 768 values)
- Positional embeddings learned for each fixed position

dHT produces:
- Variable number of tokens per image
- Tokens of different sizes
- Tokens at arbitrary positions

## The Solution: Adaptive Embedding

The `dHTEmbedder` and `dHTExtractor` solve this by:
1. **Extracting features** from variable-sized regions
2. **Interpolating positional embeddings** based on token locations
3. **Projecting** to the transformer's embedding dimension

This maintains compatibility while allowing adaptive tokenization!

In [ ]:
import torch
import torch.nn as nn
from dht.tok.tokenizer import dHTTokenizer
from dht.tok.extractor import dHTExtractor
from dht.tok.embedder import dHTEmbedder
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. The Three-Stage Pipeline

Let's understand each stage:

In [ ]:
# Create sample image
img = torch.randn(1, 3, 224, 224).to(device)

# Stage 1: Tokenization
tokenizer = dHTTokenizer(in_ch=3, hid_ch=8, compute_grad=True).to(device)
tokenizer.eval()
with torch.no_grad():
    tok_result = tokenizer(img)

print("Stage 1 - Tokenization:")
print(f"  Output: {tok_result.nV} tokens")
print(f"  Features shape: {tok_result.fV.shape}")
print(f"  Segmentation: {tok_result.seg.shape}")

In [ ]:
# Stage 2: Feature Extraction
patch_size = 16
extractor = dHTExtractor(patch_size=patch_size, channels=3).to(device)
extractor.eval()
with torch.no_grad():
    ext_result = extractor(tok_result)

print("\nStage 2 - Extraction:")
print(f"  Patch size: {patch_size}")
print(f"  Output features: {ext_result.fV.shape}")
print(f"  Feature dimension breakdown:")
print(f"    - RGB: {3 * patch_size**2} values")
print(f"    - Position: {patch_size**2} values")
if tok_result.grad is not None:
    print(f"    - Gradient: {patch_size**2} values")
print(f"    - Total: {ext_result.fV.shape[-1]} values per token")

In [ ]:
# Stage 3: Embedding
embed_dim = 384
embedder = dHTEmbedder(
    embed_dim=embed_dim,
    patch_size=patch_size,
    channels=3,
    compute_grad=True,
    num_cls_tokens=1
).to(device)
embedder.eval()
with torch.no_grad():
    emb_result = embedder(ext_result)

print("\nStage 3 - Embedding:")
print(f"  Target dimension: {embed_dim}")
print(f"  Output shape: {emb_result.fV.shape}")
print(f"  Ready for transformer!")

## 2. How Feature Extraction Works

The `dHTExtractor` does something clever:

1. **Downsample** the image to match patch_size
2. **Extract patch features** for each token's region
3. **Add positional encoding** based on region center
4. **Concatenate** RGB + position + gradient features

In [ ]:
def visualize_token_features(tok_result, ext_result, token_id=0):
    """Visualize what features a token contains."""
    
    # Get token region
    seg = tok_result.seg[0].cpu().numpy()
    token_mask = (seg == token_id)
    
    # Get features
    features = ext_result.fV[token_id].cpu().numpy()
    
    # Parse feature components
    ps = 16  # patch_size
    rgb_feats = features[:3*ps*ps]
    pos_feats = features[3*ps*ps:3*ps*ps+ps*ps]
    
    # Reshape
    rgb_patch = rgb_feats.reshape(3, ps, ps)
    pos_patch = pos_feats.reshape(ps, ps)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Token region
    axes[0].imshow(token_mask, cmap='gray')
    axes[0].set_title(f"Token {token_id} Region")
    axes[0].axis('off')
    
    # RGB features
    axes[1].imshow(rgb_patch.transpose(1, 2, 0))
    axes[1].set_title(f"RGB Features ({ps}x{ps})")
    axes[1].axis('off')
    
    # Position features
    im = axes[2].imshow(pos_patch, cmap='viridis')
    axes[2].set_title(f"Position Features ({ps}x{ps})")
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

# Visualize a token
visualize_token_features(tok_result, ext_result, token_id=5)

## 3. Positional Embedding Interpolation

A key innovation is **interpolating positional embeddings**.

Standard ViT: Learned embeddings for each of 196 fixed positions

dHT: Interpolate embeddings based on token center coordinates

In [ ]:
# Show how position encoding works
def show_position_encoding():
    H, W = 224, 224
    ps = 16
    
    # Create position grid
    y = torch.linspace(0, 1, H)
    x = torch.linspace(0, 1, W)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    
    # Downsample to patch size
    yy_down = torch.nn.functional.interpolate(
        yy.unsqueeze(0).unsqueeze(0),
        size=(ps, ps),
        mode='bilinear'
    )[0, 0]
    xx_down = torch.nn.functional.interpolate(
        xx.unsqueeze(0).unsqueeze(0),
        size=(ps, ps),
        mode='bilinear'
    )[0, 0]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(yy_down, cmap='viridis')
    axes[0].set_title(f'Y-Position Encoding ({ps}x{ps})')
    axes[0].axis('off')
    
    axes[1].imshow(xx_down, cmap='viridis')
    axes[1].set_title(f'X-Position Encoding ({ps}x{ps})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

show_position_encoding()

## 4. Handling Variable Resolution

The embedder can **interpolate** to handle different input resolutions:

In [ ]:
# Test with different image sizes
sizes = [224, 384, 512]

for size in sizes:
    test_img = torch.randn(1, 3, size, size).to(device)
    
    with torch.no_grad():
        tok_res = tokenizer(test_img)
        ext_res = extractor(tok_res)
        emb_res = embedder(ext_res)
    
    print(f"Image size {size}x{size}:")
    print(f"  Tokens: {tok_res.nV}")
    print(f"  Embedding shape: {emb_res.fV.shape}")

## 5. The Mean Injection Trick

dHT uses "mean injection" to preserve pixel-level information:

```python
replaced_mean = token_features - region_mean_color
pixel_features = original_pixel_color + replaced_mean
```

This allows the model to:
- Keep original pixel colors
- Add learned features from the tokenizer
- Maintain gradient flow

In [ ]:
# Demonstrate mean injection
from dht.utils.scatter import scatter_mean_2d

# Get original pixel colors
pixel_colors = img.permute(0, 2, 3, 1).reshape(-1, 3)

# Get token features
token_features = tok_result.fV[:, :3]  # RGB part only

# Compute region means
region_means = scatter_mean_2d(pixel_colors, tok_result.seg)

# Mean injection
replaced_mean = token_features - region_means
injected_features = pixel_colors + replaced_mean[tok_result.seg.view(-1)]

print("Mean Injection:")
print(f"  Original pixel colors: {pixel_colors.shape}")
print(f"  Token features: {token_features.shape}")
print(f"  Region means: {region_means.shape}")
print(f"  Injected features: {injected_features.shape}")
print(f"\nThis preserves pixel colors while adding learned features!")

## 6. Freezing and Unfreezing Embeddings

For fine-tuning, you can freeze non-positional parts of the embedding:

In [ ]:
# Freeze non-positional embeddings
embedder.freeze_non_pos_embed()

print("Freezing configuration:")
for name, param in embedder.named_parameters():
    print(f"  {name}: requires_grad={param.requires_grad}")

## 7. Interpolating to Different Patch Sizes

You can adapt pre-trained models to different patch sizes:

In [ ]:
# Original setup
print(f"Original pos_patch_size: {embedder.pos_patch_size}")
print(f"Original token_dim: {embedder.token_dim}")

# Interpolate to new size
new_pos_size = 32
embedder.interpolate_pos_embed(new_pos_size)

print(f"\nAfter interpolation:")
print(f"New pos_patch_size: {embedder.pos_patch_size}")
print(f"New token_dim: {embedder.token_dim}")

# Test it
with torch.no_grad():
    test_result = embedder(ext_result)
print(f"\nOutput shape: {test_result.fV.shape}")

## Summary

The "embedding fix" in dHT involves:

1. **dHTExtractor**: Extracts patch-like features from variable regions
2. **Positional encoding**: Interpolated based on token centers
3. **Mean injection**: Preserves pixel-level information
4. **dHTEmbedder**: Projects to transformer dimension
5. **Interpolation**: Adapts to different resolutions and patch sizes

### Key Benefits:
- Backward compatible with ViT architectures
- Handles variable token counts
- Resolution-agnostic
- Preserves gradient flow
- Can freeze/unfreeze components for fine-tuning

### Next: Adapting Existing ViT Models (Notebook 05)